In [30]:
import math, os, random, cv2, numpy, torch
import torch.nn as nn

In [31]:
class Conv(nn.Module):
    def __init__(self,in_channels, out_channels,kernel_size=3,stride=1,padding=1,groups=1,activation=True):
        super().__init__()
        self.conv=nn.Conv2d(in_channels,out_channels,kernel_size,stride,padding,bias=False,groups=groups)
        self.bn=nn.BatchNorm2d(out_channels,eps=0.001,momentum=0.03)
        self.act=nn.SiLU(inplace=True) if activation else nn.Identity()

    def forward(self,x):
        return self.act(self.bn(self.conv(x)))

In [32]:
class Bottleneck(nn.Module):
    def __init__(self,in_channels,out_channels,shortcut=True):
        super().__init__()
        self.conv1=Conv(in_channels,out_channels,kernel_size=3,stride=1,padding=1)
        self.conv2=Conv(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
        self.shortcut=shortcut

    def forward(self,x):
        x_in=x # for residual connection
        x=self.conv1(x)
        x=self.conv2(x)
        if self.shortcut:
            x=x+x_in
        return x
    

class C2f(nn.Module):
    def __init__(self,in_channels,out_channels, num_bottlenecks,shortcut=True):
        super().__init__()
        
        self.mid_channels=out_channels//2
        self.num_bottlenecks=num_bottlenecks

        self.conv1=Conv(in_channels,out_channels,kernel_size=1,stride=1,padding=0)
        
        # sequence of bottleneck layers
        self.m=nn.ModuleList([Bottleneck(self.mid_channels,self.mid_channels) for _ in range(num_bottlenecks)])

        self.conv2=Conv((num_bottlenecks+2)*out_channels//2,out_channels,kernel_size=1,stride=1,padding=0)
    
    def forward(self,x):
        x=self.conv1(x)

        # split x along channel dimension
        x1,x2=x[:,:x.shape[1]//2,:,:], x[:,x.shape[1]//2:,:,:]
        
        # list of outputs
        outputs=[x1,x2] # x1 is fed through the bottlenecks

        for i in range(self.num_bottlenecks):
            x1=self.m[i](x1)    # [bs,0.5c_out,w,h]
            outputs.insert(0,x1)

        outputs=torch.cat(outputs,dim=1) # [bs,0.5c_out(num_bottlenecks+2),w,h]
        out=self.conv2(outputs)

        return out
         
# sanity check
c2f=C2f(in_channels=64,out_channels=128,num_bottlenecks=2)
print(f"{sum(p.numel() for p in c2f.parameters())/1e6} million parameters")

dummy_input=torch.rand((1,64,244,244))
dummy_input=c2f(dummy_input)
print("Output shape: ", dummy_input.shape)


0.18944 million parameters
Output shape:  torch.Size([1, 128, 244, 244])


In [33]:
class SPPF(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=5):
        #kernel_size= size of maxpool
        super().__init__()
        hidden_channels=in_channels//2
        self.conv1=Conv(in_channels,hidden_channels,kernel_size=1,stride=1,padding=0)
        # concatenate outputs of maxpool and feed to conv2
        self.conv2=Conv(4*hidden_channels,out_channels,kernel_size=1,stride=1,padding=0)

        # maxpool is applied at 3 different sacles
        self.m=nn.MaxPool2d(kernel_size=kernel_size,stride=1,padding=kernel_size//2,dilation=1,ceil_mode=False)
    
    def forward(self,x):
        x=self.conv1(x)

        # apply maxpooling at diffent scales
        y1=self.m(x)
        y2=self.m(y1)
        y3=self.m(y2)

        # concantenate 
        y=torch.cat([x,y1,y2,y3],dim=1)

        # final conv
        y=self.conv2(y)

        return y

# sanity check
sppf=SPPF(in_channels=128,out_channels=512)
print(f"{sum(p.numel() for p in sppf.parameters())/1e6} million parameters")

dummy_input=sppf(dummy_input)
print("Output shape: ", dummy_input.shape)


0.140416 million parameters
Output shape:  torch.Size([1, 512, 244, 244])


In [34]:
# return d,w,r based on version
def yolo_params(version):
    if version=='n':
        return 1/3,1/4,2.0
    elif version=='s':
        return 1/3,1/2,2.0
    elif version=='m':
        return 2/3,3/4,1.5
    elif version=='l':
        return 1.0,1.0,1.0
    elif version=='x':
        return 1.0,1.25,1.0
    
class Backbone(nn.Module):
    def __init__(self,version,in_channels=3,shortcut=True):
        super().__init__()
        d,w,r=yolo_params(version)

        # conv layers
        self.conv_0=Conv(in_channels,int(64*w),kernel_size=3,stride=2,padding=1)
        self.conv_1=Conv(int(64*w),int(128*w),kernel_size=3,stride=2,padding=1)
        self.conv_3=Conv(int(128*w),int(256*w),kernel_size=3,stride=2,padding=1)
        self.conv_5=Conv(int(256*w),int(512*w),kernel_size=3,stride=2,padding=1)
        self.conv_7=Conv(int(512*w),int(512*w*r),kernel_size=3,stride=2,padding=1)

        # c2f layers
        self.c2f_2=C2f(int(128*w),int(128*w),num_bottlenecks=int(3*d),shortcut=True)
        self.c2f_4=C2f(int(256*w),int(256*w),num_bottlenecks=int(6*d),shortcut=True)
        self.c2f_6=C2f(int(512*w),int(512*w),num_bottlenecks=int(6*d),shortcut=True)
        self.c2f_8=C2f(int(512*w*r),int(512*w*r),num_bottlenecks=int(3*d),shortcut=True)

        # sppf
        self.sppf=SPPF(int(512*w*r),int(512*w*r))
    
    def forward(self,x):
        x=self.conv_0(x)
        x=self.conv_1(x)

        x=self.c2f_2(x)

        x=self.conv_3(x)

        out1=self.c2f_4(x) # keep for output

        x=self.conv_5(out1)

        out2=self.c2f_6(x) # keep for output

        x=self.conv_7(out2)
        x=self.c2f_8(x)
        out3=self.sppf(x)

        return out1,out2,out3

print("----Small model -----")
backbone_s=Backbone(version='s')
print(f"{sum(p.numel() for p in backbone_s.parameters())/1e6} million parameters")
        

        


----Small model -----
5.079712 million parameters


In [35]:
# sanity check
x=torch.rand((1,3,640,640))
out1,out2,out3=backbone_s(x)
print(out1.shape)
print(out2.shape)
print(out3.shape)

torch.Size([1, 128, 80, 80])
torch.Size([1, 256, 40, 40])
torch.Size([1, 512, 20, 20])


In [36]:
# upsample = nearest-neighbor interpolation with scale_factor=2
#            doesn't have trainable paramaters
class Upsample(nn.Module):
    def __init__(self,scale_factor=2,mode='nearest'):
        super().__init__()
        self.scale_factor=scale_factor
        self.mode=mode

    def forward(self,x):
        return nn.functional.interpolate(x,scale_factor=self.scale_factor,mode=self.mode)

In [37]:
class Neck(nn.Module):
    def __init__(self,version):
        super().__init__()
        d,w,r=yolo_params(version)

        self.up=Upsample() # no trainable parameters
        self.c2f_1=C2f(in_channels=int(512*w*(1+r)), out_channels=int(512*w),num_bottlenecks=int(3*d),shortcut=False)
        self.c2f_2=C2f(in_channels=int(768*w), out_channels=int(256*w),num_bottlenecks=int(3*d),shortcut=False)
        self.c2f_3=C2f(in_channels=int(768*w), out_channels=int(512*w),num_bottlenecks=int(3*d),shortcut=False)
        self.c2f_4=C2f(in_channels=int(512*w*(1+r)), out_channels=int(512*w*r),num_bottlenecks=int(3*d),shortcut=False)

        self.cv_1=Conv(in_channels=int(256*w),out_channels=int(256*w),kernel_size=3,stride=2, padding=1)
        self.cv_2=Conv(in_channels=int(512*w),out_channels=int(512*w),kernel_size=3,stride=2, padding=1)


    def forward(self,x_res_1,x_res_2,x):    
        # x_res_1,x_res_2,x = output of backbone
        res_1=x              # for residual connection
        
        x=self.up(x)
        x=torch.cat([x,x_res_2],dim=1)

        res_2=self.c2f_1(x)  # for residual connection
        
        x=self.up(res_2)
        x=torch.cat([x,x_res_1],dim=1)

        out_1=self.c2f_2(x)

        x=self.cv_1(out_1)

        x=torch.cat([x,res_2],dim=1)
        out_2=self.c2f_3(x)

        x=self.cv_2(out_2)

        x=torch.cat([x,res_1],dim=1)
        out_3=self.c2f_4(x)

        return out_1,out_2,out_3
    
# sanity check
neck=Neck(version='s')
print(f"{sum(p.numel() for p in neck.parameters())/1e6} million parameters")

x=torch.rand((1,3,640,640))
out1,out2,out3=Backbone(version='s')(x)
out_1,out_2,out_3=neck(out1,out2,out3)
print(out_1.shape)
print(out_2.shape)
print(out_3.shape)



3.93984 million parameters
torch.Size([1, 128, 80, 80])
torch.Size([1, 256, 40, 40])
torch.Size([1, 512, 20, 20])


In [ ]:
# DFL
class DFL(nn.Module):
    def __init__(self,ch=16):
        super().__init__()
        
        self.ch=ch
        
        self.conv=nn.Conv2d(in_channels=ch,out_channels=1,kernel_size=1,bias=False).requires_grad_(False)
        
        # initialize conv with [0,...,ch-1]
        x=torch.arange(ch,dtype=torch.float).view(1,ch,1,1)
        self.conv.weight.data[:]=torch.nn.Parameter(x) # DFL only has ch parameters

    def forward(self,x):
        # x must have num_channels = 4*ch: x=[bs,4*ch,c]
        b,c,a=x.shape                           # c=4*ch
        x=x.view(b,4,self.ch,a).transpose(1,2)  # [bs,ch,4,a]

        # take softmax on channel dimension to get distribution probabilities
        x=x.softmax(1)                          # [b,ch,4,a]
        x=self.conv(x)                          # [b,1,4,a]
        return x.view(b,4,a)                    # [b,4,a]

# sanity check
dummy_input=torch.rand((1,64,128))
dfl=DFL()
print(f"{sum(p.numel() for p in dfl.parameters())} parameters")

dummy_output=dfl(dummy_input)
print(dummy_output.shape)

print(dfl)

16 parameters
torch.Size([1, 4, 128])
DFL(
  (conv): Conv2d(16, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
)


In [ ]:
class Head(nn.Module):
    def __init__(self,version,ch=16,num_classes=80):

        super().__init__()
        self.ch=ch                          # dfl channels
        self.coordinates=self.ch*4          # number of bounding box coordinates 
        self.nc=num_classes                 # 80 for COCO
        self.no=self.coordinates+self.nc    # number of outputs per anchor box

        self.stride=torch.zeros(3)          # strides computed during build
        
        d,w,r=yolo_params(version=version)
        
        # for bounding boxes
        self.box=nn.ModuleList([
            nn.Sequential(Conv(int(256*w),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w*r),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1))
        ])

        # for classification
        self.cls=nn.ModuleList([
            nn.Sequential(Conv(int(256*w),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w*r),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1))
        ])

        # dfl
        self.dfl=DFL()

    def forward(self,x):
        # x = output of Neck = list of 3 tensors with different resolution and different channel dim
        #     x[0]=[bs, ch0, w0, h0], x[1]=[bs, ch1, w1, h1], x[2]=[bs,ch2, w2, h2] 

        for i in range(len(self.box)):       # detection head i
            box=self.box[i](x[i])            # [bs,num_coordinates,w,h]
            cls=self.cls[i](x[i])            # [bs,num_classes,w,h]
            x[i]=torch.cat((box,cls),dim=1)  # [bs,num_coordinates+num_classes,w,h]

        # in training, no dfl output
        if self.training:
            return x                         # [3,bs,num_coordinates+num_classes,w,h]
        
        # in inference time, dfl produces refined bounding box coordinates
        anchors, strides = (i.transpose(0, 1) for i in self.make_anchors(x, self.stride))

        # concatenate predictions from all detection layers
        x = torch.cat([i.view(x[0].shape[0], self.no, -1) for i in x], dim=2) #[bs, 4*self.ch + self.nc, sum_i(h[i]w[i])]
        
        # split out predictions for box and cls
        #           box=[bs,4×self.ch,sum_i(h[i]w[i])]
        #           cls=[bs,self.nc,sum_i(h[i]w[i])]
        box, cls = x.split(split_size=(4 * self.ch, self.nc), dim=1)


        a, b = self.dfl(box).chunk(2, 1)  # a=b=[bs,2×self.ch,sum_i(h[i]w[i])]
        a = anchors.unsqueeze(0) - a
        b = anchors.unsqueeze(0) + b
        box = torch.cat(tensors=((a + b) / 2, b - a), dim=1)
        
        return torch.cat(tensors=(box * strides, cls.sigmoid()), dim=1)


    def make_anchors(self, x, strides, offset=0.5):
        # x= list of feature maps: x=[x[0],...,x[N-1]], in our case N= num_detection_heads=3
        #                          each having shape [bs,ch,w,h]
        #    each feature map x[i] gives output[i] = w*h anchor coordinates + w*h stride values
        
        # strides = list of stride values indicating how much 
        #           the spatial resolution of the feature map is reduced compared to the original image

        assert x is not None
        anchor_tensor, stride_tensor = [], []
        dtype, device = x[0].dtype, x[0].device
        for i, stride in enumerate(strides):
            _, _, h, w = x[i].shape
            sx = torch.arange(end=w, device=device, dtype=dtype) + offset  # x coordinates of anchor centers
            sy = torch.arange(end=h, device=device, dtype=dtype) + offset  # y coordinates of anchor centers
            sy, sx = torch.meshgrid(sy, sx)                                # all anchor centers 
            anchor_tensor.append(torch.stack((sx, sy), -1).view(-1, 2))
            stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device))
        return torch.cat(anchor_tensor), torch.cat(stride_tensor)
        

In [ ]:
detect=Head(version='s')
print(f"{sum(p.numel() for p in detect.parameters())/1e6} million parameters")

# out_1,out_2,out_3 are output of the neck
output=detect([out_1,out_2,out_3])
print(output[0].shape)
print(output[1].shape)
print(output[2].shape)

print(detect)

In [ ]:
class CustomYolo(nn.Module):
    def __init__(self,version):
        super().__init__()
        self.backbone=Backbone(version=version)
        self.neck=Neck(version=version)
        self.head=Head(version=version)

    def forward(self,x):
        x=self.backbone(x)              # return out1,out2,out3
        x=self.neck(x[0],x[1],x[2])     # return out_1, out_2,out_3
        return self.head(list(x))
    
model=CustomYolo(version='s')
print(f"{sum(p.numel() for p in model.parameters())/1e6} million parameters")
print(model)

In [ ]:
import math
import os
import random

import cv2
import numpy
import torch
from PIL import Image
from torch.utils import data

FORMATS = 'bmp', 'dng', 'jpeg', 'jpg', 'mpo', 'png', 'tif', 'tiff', 'webp'


class Dataset(data.Dataset):
    def __init__(self, filenames, input_size, params, augment):
        self.params = params
        self.mosaic = augment
        self.augment = augment
        self.input_size = input_size

        # Read labels
        labels = self.load_label(filenames)
        self.labels = list(labels.values())
        self.filenames = list(labels.keys())  # update
        self.n = len(self.filenames)  # number of samples
        self.indices = range(self.n)
        # Albumentations (optional, only used if package is installed)
        self.albumentations = Albumentations()

    def __getitem__(self, index):
        index = self.indices[index]

        params = self.params
        mosaic = self.mosaic and random.random() < params['mosaic']

        if mosaic:
            # Load MOSAIC
            image, label = self.load_mosaic(index, params)
            # MixUp augmentation
            if random.random() < params['mix_up']:
                index = random.choice(self.indices)
                mix_image1, mix_label1 = image, label
                mix_image2, mix_label2 = self.load_mosaic(index, params)

                image, label = mix_up(mix_image1, mix_label1, mix_image2, mix_label2)
        else:
            # Load image
            image, shape = self.load_image(index)
            h, w = image.shape[:2]

            # Resize
            image, ratio, pad = resize(image, self.input_size, self.augment)

            label = self.labels[index].copy()
            if label.size:
                label[:, 1:] = wh2xy(label[:, 1:], ratio[0] * w, ratio[1] * h, pad[0], pad[1])
            if self.augment:
                image, label = random_perspective(image, label, params)

        nl = len(label)  # number of labels
        h, w = image.shape[:2]
        cls = label[:, 0:1]
        box = label[:, 1:5]
        box = xy2wh(box, w, h)

        if self.augment:
            # Albumentations
            image, box, cls = self.albumentations(image, box, cls)
            nl = len(box)  # update after albumentations
            # HSV color-space
            augment_hsv(image, params)
            # Flip up-down
            if random.random() < params['flip_ud']:
                image = numpy.flipud(image)
                if nl:
                    box[:, 1] = 1 - box[:, 1]
            # Flip left-right
            if random.random() < params['flip_lr']:
                image = numpy.fliplr(image)
                if nl:
                    box[:, 0] = 1 - box[:, 0]

        target_cls = torch.zeros((nl, 1))
        target_box = torch.zeros((nl, 4))
        if nl > 0:
            target_cls = torch.from_numpy(cls).view(-1, 1)
            target_box = torch.from_numpy(box).view(-1, 4)
        else:
            # nl == 0 case
            target_cls = torch.zeros((0, 1)) 
            target_box = torch.zeros((0, 4))

        # Convert HWC to CHW, BGR to RGB
        sample = image.transpose((2, 0, 1))[::-1]
        sample = numpy.ascontiguousarray(sample)

        return torch.from_numpy(sample), target_cls, target_box, torch.zeros((nl, 1))

    def __len__(self):
        return len(self.filenames)

    def load_image(self, i):
        try:
            image = cv2.imread(self.filenames[i])
            if image is None:
                raise ValueError(f"OpenCV failed to load {self.filenames[i]}")
            h, w = image.shape[:2]
            r = self.input_size / max(h, w)
            if r != 1:
                image = cv2.resize(image,
                                   dsize=(int(w * r), int(h * r)),
                                   interpolation=resample() if self.augment else cv2.INTER_LINEAR)
            return image, (h, w)
        except Exception as e:
            print(f"Warning: Skipping corrupted image {self.filenames[i]}: {e}")
            # Return a black image
            return numpy.zeros((self.input_size, self.input_size, 3), dtype=numpy.uint8), (0, 0)

    def load_mosaic(self, index, params):
        label4 = []
        border = [-self.input_size // 2, -self.input_size // 2]
        image4 = numpy.full((self.input_size * 2, self.input_size * 2, 3), 0, dtype=numpy.uint8)
        y1a, y2a, x1a, x2a, y1b, y2b, x1b, x2b = (None, None, None, None, None, None, None, None)

        xc = int(random.uniform(-border[0], 2 * self.input_size + border[1]))
        yc = int(random.uniform(-border[0], 2 * self.input_size + border[1]))

        indices = [index] + random.choices(self.indices, k=3)
        random.shuffle(indices)

        for i, index in enumerate(indices):
            # Load image
            image, _ = self.load_image(index)
            shape = image.shape
            if i == 0:  # top left
                x1a = max(xc - shape[1], 0)
                y1a = max(yc - shape[0], 0)
                x2a = xc
                y2a = yc
                x1b = shape[1] - (x2a - x1a)
                y1b = shape[0] - (y2a - y1a)
                x2b = shape[1]
                y2b = shape[0]
            if i == 1:  # top right
                x1a = xc
                y1a = max(yc - shape[0], 0)
                x2a = min(xc + shape[1], self.input_size * 2)
                y2a = yc
                x1b = 0
                y1b = shape[0] - (y2a - y1a)
                x2b = min(shape[1], x2a - x1a)
                y2b = shape[0]
            if i == 2:  # bottom left
                x1a = max(xc - shape[1], 0)
                y1a = yc
                x2a = xc
                y2a = min(self.input_size * 2, yc + shape[0])
                x1b = shape[1] - (x2a - x1a)
                y1b = 0
                x2b = shape[1]
                y2b = min(y2a - y1a, shape[0])
            if i == 3:  # bottom right
                x1a = xc
                y1a = yc
                x2a = min(xc + shape[1], self.input_size * 2)
                y2a = min(self.input_size * 2, yc + shape[0])
                x1b = 0
                y1b = 0
                x2b = min(shape[1], x2a - x1a)
                y2b = min(y2a - y1a, shape[0])

            pad_w = x1a - x1b
            pad_h = y1a - y1b
            image4[y1a:y2a, x1a:x2a] = image[y1b:y2b, x1b:x2b]

            # Labels
            label = self.labels[index].copy()
            if len(label):
                label[:, 1:] = wh2xy(label[:, 1:], shape[1], shape[0], pad_w, pad_h)
            label4.append(label)

        # Concat/clip labels
        label4 = numpy.concatenate(label4, 0)
        for x in label4[:, 1:]:
            numpy.clip(x, 0, 2 * self.input_size, out=x)

        # Augment
        image4, label4 = random_perspective(image4, label4, params, border)

        return image4, label4

    @staticmethod
    def collate_fn(batch):
        samples, cls, box, _ = zip(*batch)
    
        # Stack the images into a 4D tensor [B, C, H, W]
        samples = torch.stack(samples, dim=0)

        # We only concatenate if the tensor has a size > 0
        valid_cls = [c.view(-1, 1) for c in cls if c.shape[0] > 0]
        valid_box = [b.view(-1, 4) for b in box if b.shape[0] > 0]
        
        batch_idx = []
        if valid_cls:
            # Concatenate the labels
            cls_out = torch.cat(valid_cls, dim=0)
            box_out = torch.cat(valid_box, dim=0)
            
            # Rebuild the batch index (idx)
            for i, c in enumerate(cls):
                nl = c.shape[0]
                if nl > 0:
                    batch_idx.append(torch.full((nl,), i, dtype=torch.float32))
            idx_out = torch.cat(batch_idx, dim=0)
        else:
            # Entire batch is background images
            cls_out = torch.zeros((0, 1))
            box_out = torch.zeros((0, 4))
            idx_out = torch.zeros(0)

        targets = {
            'cls': cls_out,
            'box': box_out,
            'idx': idx_out
        }
        return samples, targets

    @staticmethod
    def load_label(filenames):
        path = f'{os.path.dirname(filenames[0])}.cache'
        if os.path.exists(path):
            return torch.load(path, weights_only=False)
        x = {}
        for filename in filenames:
            try:
                # verify images
                with open(filename, 'rb') as f:
                    image = Image.open(f)
                    image.verify()  # PIL verify
                shape = image.size  # image size
                assert (shape[0] > 9) & (shape[1] > 9), f'image size {shape} <10 pixels'
                assert image.format.lower() in FORMATS, f'invalid image format {image.format}'

                # verify labels
                a = f'{os.sep}images{os.sep}'
                b = f'{os.sep}labels{os.sep}'
                if os.path.isfile(b.join(filename.rsplit(a, 1)).rsplit('.', 1)[0] + '.txt'):
                    with open(b.join(filename.rsplit(a, 1)).rsplit('.', 1)[0] + '.txt') as f:
                        label = [x.split() for x in f.read().strip().splitlines() if len(x)]
                        label = numpy.array(label, dtype=numpy.float32)
                    nl = len(label)
                    if nl:
                        assert (label >= 0).all()
                        assert label.shape[1] == 5
                        assert (label[:, 1:] <= 1).all()
                        _, i = numpy.unique(label, axis=0, return_index=True)
                        if len(i) < nl:  # duplicate row check
                            label = label[i]  # remove duplicates
                    else:
                        label = numpy.zeros((0, 5), dtype=numpy.float32)
                else:
                    label = numpy.zeros((0, 5), dtype=numpy.float32)
            except FileNotFoundError:
                label = numpy.zeros((0, 5), dtype=numpy.float32)
            except AssertionError:
                continue
            x[filename] = label
        torch.save(x, path)
        return x


def wh2xy(x, w=640, h=640, pad_w=0, pad_h=0):
    # Convert nx4 boxes
    # from [x, y, w, h] normalized to [x1, y1, x2, y2] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else numpy.copy(x)
    y[:, 0] = w * (x[:, 0] - x[:, 2] / 2) + pad_w  # top left x
    y[:, 1] = h * (x[:, 1] - x[:, 3] / 2) + pad_h  # top left y
    y[:, 2] = w * (x[:, 0] + x[:, 2] / 2) + pad_w  # bottom right x
    y[:, 3] = h * (x[:, 1] + x[:, 3] / 2) + pad_h  # bottom right y
    return y


def xy2wh(x, w, h):
    # warning: inplace clip
    x[:, [0, 2]] = x[:, [0, 2]].clip(0, w - 1E-3)  # x1, x2
    x[:, [1, 3]] = x[:, [1, 3]].clip(0, h - 1E-3)  # y1, y2

    # Convert nx4 boxes
    # from [x1, y1, x2, y2] to [x, y, w, h] normalized where xy1=top-left, xy2=bottom-right
    y = numpy.copy(x)
    y[:, 0] = ((x[:, 0] + x[:, 2]) / 2) / w  # x center
    y[:, 1] = ((x[:, 1] + x[:, 3]) / 2) / h  # y center
    y[:, 2] = (x[:, 2] - x[:, 0]) / w  # width
    y[:, 3] = (x[:, 3] - x[:, 1]) / h  # height
    return y


def resample():
    choices = (cv2.INTER_AREA,
               cv2.INTER_CUBIC,
               cv2.INTER_LINEAR,
               cv2.INTER_NEAREST,
               cv2.INTER_LANCZOS4)
    return random.choice(seq=choices)


def augment_hsv(image, params):
    # HSV color-space augmentation
    h = params['hsv_h']
    s = params['hsv_s']
    v = params['hsv_v']

    r = numpy.random.uniform(-1, 1, 3) * [h, s, v] + 1
    h, s, v = cv2.split(cv2.cvtColor(image, cv2.COLOR_BGR2HSV))

    x = numpy.arange(0, 256, dtype=r.dtype)
    lut_h = ((x * r[0]) % 180).astype('uint8')
    lut_s = numpy.clip(x * r[1], 0, 255).astype('uint8')
    lut_v = numpy.clip(x * r[2], 0, 255).astype('uint8')

    hsv = cv2.merge((cv2.LUT(h, lut_h), cv2.LUT(s, lut_s), cv2.LUT(v, lut_v)))
    cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR, dst=image)  # no return needed


def resize(image, input_size, augment):
    # Resize and pad image while meeting stride-multiple constraints
    shape = image.shape[:2]  # current shape [height, width]

    # Scale ratio (new / old)
    r = min(input_size / shape[0], input_size / shape[1])
    if not augment:  # only scale down, do not scale up (for better val mAP)
        r = min(r, 1.0)

    # Compute padding
    pad = int(round(shape[1] * r)), int(round(shape[0] * r))
    w = (input_size - pad[0]) / 2
    h = (input_size - pad[1]) / 2

    if shape[::-1] != pad:  # resize
        image = cv2.resize(image,
                           dsize=pad,
                           interpolation=resample() if augment else cv2.INTER_LINEAR)
    top, bottom = int(round(h - 0.1)), int(round(h + 0.1))
    left, right = int(round(w - 0.1)), int(round(w + 0.1))
    image = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT)  # add border
    return image, (r, r), (w, h)


def candidates(box1, box2):
    # box1(4,n), box2(4,n)
    w1, h1 = box1[2] - box1[0], box1[3] - box1[1]
    w2, h2 = box2[2] - box2[0], box2[3] - box2[1]
    aspect_ratio = numpy.maximum(w2 / (h2 + 1e-16), h2 / (w2 + 1e-16))  # aspect ratio
    return (w2 > 2) & (h2 > 2) & (w2 * h2 / (w1 * h1 + 1e-16) > 0.1) & (aspect_ratio < 100)


def random_perspective(image, label, params, border=(0, 0)):
    h = image.shape[0] + border[0] * 2
    w = image.shape[1] + border[1] * 2

    # Center
    center = numpy.eye(3)
    center[0, 2] = -image.shape[1] / 2  # x translation (pixels)
    center[1, 2] = -image.shape[0] / 2  # y translation (pixels)

    # Perspective
    perspective = numpy.eye(3)

    # Rotation and Scale
    rotate = numpy.eye(3)
    a = random.uniform(-params['degrees'], params['degrees'])
    s = random.uniform(1 - params['scale'], 1 + params['scale'])
    rotate[:2] = cv2.getRotationMatrix2D(angle=a, center=(0, 0), scale=s)

    # Shear
    shear = numpy.eye(3)
    shear[0, 1] = math.tan(random.uniform(-params['shear'], params['shear']) * math.pi / 180)
    shear[1, 0] = math.tan(random.uniform(-params['shear'], params['shear']) * math.pi / 180)

    # Translation
    translate = numpy.eye(3)
    translate[0, 2] = random.uniform(0.5 - params['translate'], 0.5 + params['translate']) * w
    translate[1, 2] = random.uniform(0.5 - params['translate'], 0.5 + params['translate']) * h

    # Combined rotation matrix, order of operations (right to left) is IMPORTANT
    matrix = translate @ shear @ rotate @ perspective @ center
    if (border[0] != 0) or (border[1] != 0) or (matrix != numpy.eye(3)).any():  # image changed
        image = cv2.warpAffine(image, matrix[:2], dsize=(w, h), borderValue=(0, 0, 0))

    # Transform label coordinates
    n = len(label)
    if n:
        xy = numpy.ones((n * 4, 3))
        xy[:, :2] = label[:, [1, 2, 3, 4, 1, 4, 3, 2]].reshape(n * 4, 2)  # x1y1, x2y2, x1y2, x2y1
        xy = xy @ matrix.T  # transform
        xy = xy[:, :2].reshape(n, 8)  # perspective rescale or affine

        # create new boxes
        x = xy[:, [0, 2, 4, 6]]
        y = xy[:, [1, 3, 5, 7]]
        box = numpy.concatenate((x.min(1), y.min(1), x.max(1), y.max(1))).reshape(4, n).T

        # clip
        box[:, [0, 2]] = box[:, [0, 2]].clip(0, w)
        box[:, [1, 3]] = box[:, [1, 3]].clip(0, h)
        # filter candidates
        indices = candidates(box1=label[:, 1:5].T * s, box2=box.T)

        label = label[indices]
        label[:, 1:5] = box[indices]

    return image, label


def mix_up(image1, label1, image2, label2):
    # Applies MixUp augmentation https://arxiv.org/pdf/1710.09412.pdf
    alpha = numpy.random.beta(a=32.0, b=32.0)  # mix-up ratio, alpha=beta=32.0
    image = (image1 * alpha + image2 * (1 - alpha)).astype(numpy.uint8)
    label = numpy.concatenate((label1, label2), 0)
    return image, label


class Albumentations:
    def __init__(self, size=640):
        self.transform = None
        try:
            import albumentations as A
            transforms = [
                A.HorizontalFlip(p=0.5),
                A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0), p=0.1),
                A.Blur(p=0.01),
                A.MedianBlur(p=0.01),
                A.CLAHE(p=0.01),
                A.ToGray(p=0.01),
            ]
            
            # Using BboxParams with min_visibility to filter out tiny/broken boxes
            self.transform = A.Compose(
                transforms,
                bbox_params=A.BboxParams(
                    format='yolo', 
                    label_fields=['class_labels'],
                    min_visibility=0.4 # Drops boxes if <40% remains after cropping
                )
            )
            print("Albumentations pipeline initialized successfully.")
            
        except ImportError:
            print("Albumentations not found. Skipping augmentations.")
            pass

    def __call__(self, image, box, cls):
        if self.transform and len(box) > 0:
            # Albumentations expects RGB for some transforms (like CLAHE/ToGray)
            # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 

            new = self.transform(image=image, bboxes=box, class_labels=cls)
            
            # Ensure we handle cases where transforms might remove all boxes
            if len(new['bboxes']) > 0:
                image = new['image']
                box = numpy.array(new['bboxes'], dtype=numpy.float32)
                cls = numpy.array(new['class_labels'], dtype=numpy.float32)
            
        return image, box, cls

In [ ]:
import os
import yaml
import glob
from torch.utils import data

# get all file names
data_dir= '/home/omen/AI_Tools/yolov8_detection-main/dataset'
val_split_path = os.path.join(data_dir, 'val', 'images', 'val')
split = 'train2017'
filenames_train = glob.glob(os.path.join(data_dir, 'images', split, '*.*'))
filenames_val = glob.glob(os.path.join(val_split_path, '*.*'))

# input_size for the model
input_size=640

params = {'min_lr': 0.0001, 
          'max_lr': 0.01, 
          'momentum': 0.937, 
          'weight_decay': 0.0005, 
          'warmup_epochs': 3.0, 
          'box': 7.5, 
          'cls': 0.5, 
          'dfl': 1.5, 
          'hsv_h': 0.015, 
          'hsv_s': 0.7, 
          'hsv_v': 0.4, 
          'degrees': 0.0, 
          'translate': 0.1, 
          'scale': 0.5, 
          'shear': 0.0, 
          'flip_ud': 0.0, 
          'flip_lr': 0.5, 
          'mosaic': 1.0, 
          'mix_up': 0.0, 
          'names': {0: 'dog', 1: 'person', 2: 'skateboard', 3: 'toilet', 4: 'sink', 5: 'car', 6: 'backpack', 7: 'skis', 8: 'bench', 9: 'sheep', 10: 'cell phone', 11: 'traffic light', 12: 'sports ball', 13: 'baseball glove', 14: 'tv', 15: 'chair', 16: 'bed', 17: 'couch', 18: 'horse', 19: 'truck', 20: 'bus', 21: 'tie', 22: 'handbag', 23: 'kite', 24: 'cat', 25: 'refrigerator', 26: 'bowl', 27: 'boat', 28: 'clock', 29: 'bird', 30: 'surfboard', 31: 'cup', 32: 'pizza', 33: 'dining table', 34: 'broccoli', 35: 'snowboard', 36: 'bottle', 37: 'wine glass', 38: 'spoon', 39: 'oven', 40: 'toaster', 41: 'book', 42: 'train', 43: 'microwave', 44: 'teddy bear', 45: 'suitcase', 46: 'mouse', 47: 'keyboard', 48: 'airplane', 49: 'vase', 50: 'fire hydrant', 51: 'baseball bat', 52: 'scissors', 53: 'laptop', 54: 'hot dog', 55: 'bear', 56: 'sandwich', 57: 'fork', 58: 'potted plant', 59: 'cow', 60: 'giraffe', 61: 'orange', 62: 'umbrella', 63: 'tennis racket', 64: 'banana', 65: 'cake', 66: 'carrot', 67: 'remote', 68: 'frisbee', 69: 'motorcycle', 70: 'stop sign', 71: 'apple', 72: 'donut', 73: 'toothbrush', 74: 'knife', 75: 'bicycle', 76: 'elephant', 77: 'zebra', 78: 'hair drier', 79: 'parking meter'}}

train_data=Dataset(filenames_train,input_size,params,augment=True)
train_loader = data.DataLoader(train_data, batch_size=16, num_workers=8, pin_memory=True, collate_fn=Dataset.collate_fn)
print(f"Train_loader : {len(train_loader)} batches ({len(filenames_train)} images)")

# params here should have augmentation probabilities set to 0
val_params = params.copy()
val_params.update({
    'mosaic': 0.0,
    'mix_up': 0.0,
    'flip_lr': 0.0,
    'flip_ud': 0.0,
    'augment': False,
    'names': {0: 'bottle', 1: 'person', 2: 'hot dog', 3: 'toilet', 4: 'bowl', 5: 'sink', 6: 'vase', 7: 'potted plant', 8: 'cup', 9: 'apple', 10: 'microwave', 11: 'oven', 12: 'airplane', 13: 'clock', 14: 'dining table', 15: 'pizza', 16: 'baseball bat', 17: 'baseball glove', 18: 'traffic light', 19: 'skis', 20: 'car', 21: 'fire hydrant', 22: 'kite', 23: 'sports ball', 24: 'chair', 25: 'tennis racket', 26: 'bus', 27: 'frisbee', 28: 'wine glass', 29: 'elephant', 30: 'train', 31: 'toothbrush', 32: 'cat', 33: 'book', 34: 'couch', 35: 'skateboard', 36: 'bird', 37: 'truck', 38: 'dog', 39: 'fork', 40: 'spoon', 41: 'knife', 42: 'tie', 43: 'motorcycle', 44: 'tv', 45: 'bicycle', 46: 'suitcase', 47: 'handbag', 48: 'laptop', 49: 'remote', 50: 'surfboard', 51: 'banana', 52: 'backpack', 53: 'bed', 54: 'boat', 55: 'cow', 56: 'snowboard', 57: 'giraffe', 58: 'bench', 59: 'umbrella', 60: 'cake', 61: 'teddy bear', 62: 'horse', 63: 'cell phone', 64: 'zebra', 65: 'donut', 66: 'orange', 67: 'keyboard', 68: 'broccoli', 69: 'sandwich', 70: 'carrot', 71: 'sheep', 72: 'scissors', 73: 'parking meter', 74: 'mouse', 75: 'stop sign', 76: 'bear', 77: 'refrigerator', 78: 'hair drier', 79: 'toaster'}
})

val_data = Dataset(filenames_val, input_size, params, augment=False)

val_loader = data.DataLoader(
    val_data, 
    batch_size=32, 
    num_workers=8, 
    pin_memory=True, 
    collate_fn=Dataset.collate_fn,
    shuffle=False  # Keep it False for consistent metric calculation
)
print(f"Val_loader : {len(val_loader)} batches ({len(filenames_val)} images)")



from torch.utils.data import Subset
import random

# Create indices for the first 128 images
indices = random.sample(range(49001), 1024)
train_data_subset = Subset(train_data, indices)

# Re-initialize loader using the subset
subset_loader = data.DataLoader(
    train_data_subset, 
    batch_size=32, 
    num_workers=8, 
    pin_memory=True, 
    collate_fn=Dataset.collate_fn,
    shuffle=False
)
print(f"subset_loader : {len(subset_loader)}")

In [ ]:
batch=next(iter(train_loader))
print("All keys in batch      : ", batch[1].keys())
print(f"Input batch shape      : ", batch[0].shape)
print(f"Classification scores  : {batch[1]['cls'].shape}")
print(f"Box coordinates        : {batch[1]['box'].shape}")
print(f"Index identifier (which score belongs to which image): {batch[1]['idx'].shape}")

In [ ]:
import copy
import random
from time import time

import math
import numpy
import torch
import torchvision
from torch.nn.functional import cross_entropy


def setup_seed():
    """
    Setup random seed.
    """
    random.seed(0)
    numpy.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def setup_multi_processes():
    """
    Setup multi-processing environment variables.
    """
    import cv2
    from os import environ
    from platform import system

    # set multiprocess start method as `fork` to speed up the training
    if system() != 'Windows':
        # 'forkserver' is cleaner than 'fork' for CUDA applications to avoid deadlocks
        try:
            torch.multiprocessing.set_start_method('forkserver', force=True)
        except RuntimeError:
            pass


    # disable opencv multithreading to avoid system being overloaded
    cv2.setNumThreads(0)

    # Limit threads to prevent CPU-bound heat from throttling your GPU
    environ.update({
        'OMP_NUM_THREADS': '1',
        'MKL_NUM_THREADS': '1',
        'OPENBLAS_NUM_THREADS': '1',
        'VECLIB_MAXIMUM_THREADS': '1',
        'NUMEXPR_NUM_THREADS': '1'
    })


def export_onnx(args, model, path='./weights/best.pt'):
    import onnx
    from onnxsim import simplify

    print(f"Starting ONNX export for {path}...")
    
    model.eval()
    # Fuse Conv2d + BatchNorm2d layers for faster inference
    if hasattr(model, 'fuse'):
        model = model.fuse()

    # Define I/O
    inputs = ['images']
    outputs = ['outputs']
    # Dynamic axes allow the model to take different batch sizes or even resolutions
    dynamic = {
        'images': {0: 'batch', 2: 'height', 3: 'width'},
        'outputs': {0: 'batch', 1: 'anchors'}
    }

    # Dummy input
    x = torch.zeros((1, 3, args.input_size, args.input_size)).to(next(model.parameters()).device)

    # Export
    onnx_path = path.replace('.pt', '.onnx')
    torch.onnx.export(
        model, x, onnx_path,
        verbose=False,
        opset_version=12,
        do_constant_folding=True,
        input_names=inputs,
        output_names=outputs,
        dynamic_axes=dynamic
    )

    # Simplify (Graph Optimization)
    try:
        model_onnx = onnx.load(onnx_path)
        onnx.checker.check_model(model_onnx)
        if simplify:
            model_onnx, check = simplify(model_onnx)
            assert check, "Simplified ONNX model could not be validated"
        onnx.save(model_onnx, onnx_path)
        print(f"ONNX export success: {onnx_path}")
    except Exception as e:
        print(f"ONNX simplification failed: {e}")


def make_anchors(x, strides, offset=0.5):
    assert x is not None
    anchor_tensor, stride_tensor = [], []
    dtype, device = x[0].dtype, x[0].device
    for i, stride in enumerate(strides):
        _, _, h, w = x[i].shape
        sx = torch.arange(end=w, device=device, dtype=dtype) + offset  # shift x
        sy = torch.arange(end=h, device=device, dtype=dtype) + offset  # shift y
        sy, sx = torch.meshgrid(sy, sx)
        anchor_tensor.append(torch.stack((sx, sy), -1).view(-1, 2))
        stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device))
    return torch.cat(anchor_tensor), torch.cat(stride_tensor)


def compute_metric(output, target, iou_v):
    # intersection(N,M) = (rb(N,M,2) - lt(N,M,2)).clamp(0).prod(2)
    (a1, a2) = target[:, 1:].unsqueeze(1).chunk(2, 2)
    (b1, b2) = output[:, :4].unsqueeze(0).chunk(2, 2)
    intersection = (torch.min(a2, b2) - torch.max(a1, b1)).clamp(0).prod(2)
    # IoU = intersection / (area1 + area2 - intersection)
    iou = intersection / ((a2 - a1).prod(2) + (b2 - b1).prod(2) - intersection + 1e-7)

    correct = numpy.zeros((output.shape[0], iou_v.shape[0]))
    correct = correct.astype(bool)
    for i in range(len(iou_v)):
        # IoU > threshold and classes match
        x = torch.where((iou >= iou_v[i]) & (target[:, 0:1] == output[:, 5]))
        if x[0].shape[0]:
            matches = torch.cat((torch.stack(x, 1),
                                 iou[x[0], x[1]][:, None]), 1).cpu().numpy()  # [label, detect, iou]
            if x[0].shape[0] > 1:
                matches = matches[matches[:, 2].argsort()[::-1]]
                matches = matches[numpy.unique(matches[:, 1], return_index=True)[1]]
                matches = matches[numpy.unique(matches[:, 0], return_index=True)[1]]
            correct[matches[:, 1].astype(int), i] = True
    return torch.tensor(correct, dtype=torch.bool, device=output.device)


def non_max_suppression(outputs, confidence_threshold=0.001, iou_threshold=0.7):
    max_wh = 7680
    max_det = 300
    max_nms = 30000

    bs = outputs.shape[0]  # batch size
    nc = outputs.shape[1] - 4  # number of classes
    xc = outputs[:, 4:4 + nc].amax(1) > confidence_threshold  # candidates

    # Settings
    start = time()
    limit = 0.5 + 0.05 * bs  # seconds to quit after
    output = [torch.zeros((0, 6), device=outputs.device)] * bs
    
    # for index, x in enumerate(outputs):  # image index, image inference
    #     x = x.transpose(0, -1)[xc[index]]  # confidence

    #     # If none remain process next image
    #     if not x.shape[0]:
    #         continue

    #     # matrix nx6 (box, confidence, cls)
    #     box, cls = x.split((4, nc), 1)
    #     # box = wh2xy(box)  # (cx, cy, w, h) to (x1, y1, x2, y2)
    #     box = x[:, :4]
    #     if nc > 1:
    #         i, j = (cls > confidence_threshold).nonzero(as_tuple=False).T
    #         x = torch.cat((box[i], x[i, 4 + j, None], j[:, None].float()), 1)
    #     else:  # best class only
    #         conf, j = cls.max(1, keepdim=True)
    #         x = torch.cat((box, conf, j.float()), 1)[conf.view(-1) > confidence_threshold]

    #     # Check shape
    #     n = x.shape[0]  # number of boxes
    #     if not n:  # no boxes
    #         continue
    #     x = x[x[:, 4].argsort(descending=True)[:max_nms]]  # sort by confidence and remove excess boxes

    for index, x in enumerate(outputs):  # x shape: [8400, 84]
        # 1. Filter by confidence threshold (max class score)
        # x is [anchors, 4 + nc]
        conf_scores = x[:, 4:] 
        max_scores, class_ids = conf_scores.max(1)
        mask = max_scores > confidence_threshold
        
        x = x[mask]
        if not x.shape[0]:
            continue

        # 2. Extract final 6 columns: [x1, y1, x2, y2, conf, cls]
        # We take the 4 box columns and cat the max_score and class_id
        boxes = x[:, :4]
        scores = max_scores[mask].unsqueeze(1)
        ids = class_ids[mask].unsqueeze(1).float()
        
        x = torch.cat((boxes, scores, ids), 1) # Shape: [n, 6]

        # 3. Sort by confidence and remove excess boxes (optional but faster)
        n = x.shape[0]
        if not n:
            continue
        x = x[x[:, 4].argsort(descending=True)[:max_nms]]

        # Batched NMS
        c = x[:, 5:6] * max_wh  # classes
        boxes, scores = x[:, :4] + c, x[:, 4]  # boxes, scores
        indices = torchvision.ops.nms(boxes, scores, iou_threshold)  # NMS
        indices = indices[:max_det]  # limit detections

        output[index] = x[indices]
        if (time() - start) > limit:
            break  # time limit exceeded

    return output


def smooth(y, f=0.05):
    # Box filter of fraction f
    nf = round(len(y) * f * 2) // 2 + 1  # number of filter elements (must be odd)
    p = numpy.ones(nf // 2)  # ones padding
    yp = numpy.concatenate((p * y[0], y, p * y[-1]), 0)  # y padded
    return numpy.convolve(yp, numpy.ones(nf) / nf, mode='valid')  # y-smoothed


def compute_ap(tp, conf, pred_cls, target_cls, eps=1e-16):
    """
    Compute the average precision, given the recall and precision curves.
    Source: https://github.com/rafaelpadilla/Object-Detection-Metrics.
    # Arguments
        tp:  True positives (nparray, nx1 or nx10).
        conf:  Object-ness value from 0-1 (nparray).
        pred_cls:  Predicted object classes (nparray).
        target_cls:  True object classes (nparray).
    # Returns
        The average precision
    """
    # Sort by object-ness
    i = numpy.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]

    # Find unique classes
    unique_classes, nt = numpy.unique(target_cls, return_counts=True)
    nc = unique_classes.shape[0]  # number of classes, number of detections

    # Create Precision-Recall curve and compute AP for each class
    p = numpy.zeros((nc, 1000))
    r = numpy.zeros((nc, 1000))
    ap = numpy.zeros((nc, tp.shape[1]))
    px, py = numpy.linspace(0, 1, 1000), []  # for plotting
    for ci, c in enumerate(unique_classes):
        i = pred_cls == c
        nl = nt[ci]  # number of labels
        no = i.sum()  # number of outputs
        if no == 0 or nl == 0:
            continue

        # Accumulate FPs and TPs
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)

        # Recall
        recall = tpc / (nl + eps)  # recall curve
        # negative x, xp because xp decreases
        r[ci] = numpy.interp(-px, -conf[i], recall[:, 0], left=0)

        # Precision
        precision = tpc / (tpc + fpc)  # precision curve
        p[ci] = numpy.interp(-px, -conf[i], precision[:, 0], left=1)  # p at pr_score

        # AP from recall-precision curve
        for j in range(tp.shape[1]):
            m_rec = numpy.concatenate(([0.0], recall[:, j], [1.0]))
            m_pre = numpy.concatenate(([1.0], precision[:, j], [0.0]))

            # Compute the precision envelope
            m_pre = numpy.flip(numpy.maximum.accumulate(numpy.flip(m_pre)))

            # Integrate area under curve
            x = numpy.linspace(0, 1, 101)  # 101-point interp (COCO)
            ap[ci, j] = numpy.trapz(numpy.interp(x, m_rec, m_pre), x)  # integrate

    # Compute F1 (harmonic mean of precision and recall)
    f1 = 2 * p * r / (p + r + eps)

    i = smooth(f1.mean(0), 0.1).argmax()  # max F1 index
    p, r, f1 = p[:, i], r[:, i], f1[:, i]
    tp = (r * nt).round()  # true positives
    fp = (tp / (p + eps) - tp).round()  # false positives
    ap50, ap = ap[:, 0], ap.mean(1)  # AP@0.5, AP@0.5:0.95
    m_pre, m_rec = p.mean(), r.mean()
    map50, mean_ap = ap50.mean(), ap.mean()
    return tp, fp, m_pre, m_rec, map50, mean_ap


def compute_iou(box1, box2, eps=1e-7):
    # Returns Intersection over Union (IoU) of box1(1,4) to box2(n,4)

    # Get the coordinates of bounding boxes
    b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
    b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
    w1, h1 = b1_x2 - b1_x1, b1_y2 - b1_y1 + eps
    w2, h2 = b2_x2 - b2_x1, b2_y2 - b2_y1 + eps

    # Intersection area
    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp(0) * \
            (b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)).clamp(0)

    # Union Area
    union = w1 * h1 + w2 * h2 - inter + eps

    # IoU
    iou = inter / union
    cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)  # convex (smallest enclosing box) width
    ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)  # convex height
    c2 = cw ** 2 + ch ** 2 + eps  # convex diagonal squared
    rho2 = ((b2_x1 + b2_x2 - b1_x1 - b1_x2) ** 2 + (b2_y1 + b2_y2 - b1_y1 - b1_y2) ** 2) / 4  # center dist ** 2
    # https://github.com/Zzh-tju/DIoU-SSD-pytorch/blob/master/utils/box/box_utils.py#L47
    v = (4 / math.pi ** 2) * (torch.atan(w2 / h2) - torch.atan(w1 / h1)).pow(2)
    with torch.no_grad():
        alpha = v / (v - iou + (1 + eps))
    return iou - (rho2 / c2 + v * alpha)  # CIoU


def strip_optimizer(filename):
    x = torch.load(filename, map_location="cpu")
    x['model'].half()  # to FP16
    for p in x['model'].parameters():
        p.requires_grad = False
    torch.save(x, f=filename)


def clip_gradients(model, max_norm=10.0):
    parameters = model.parameters()
    torch.nn.utils.clip_grad_norm_(parameters, max_norm=max_norm)


def load_weight(model, ckpt):
    dst = model.state_dict()
    src = torch.load(ckpt)['model'].float().cpu()

    ckpt = {}
    for k, v in src.state_dict().items():
        if k in dst and v.shape == dst[k].shape:
            ckpt[k] = v

    model.load_state_dict(state_dict=ckpt, strict=False)
    return model


def set_params(model, decay):
    p1 = []
    p2 = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim <= 1 or name.endswith(".bias"):
            p1.append(param)
        else:
            p2.append(param)
    return [{'params': p1, 'weight_decay': 0.00},
            {'params': p2, 'weight_decay': decay}]


def plot_lr(args, optimizer, scheduler, num_steps):
    from matplotlib import pyplot

    optimizer = copy.copy(optimizer)
    scheduler = copy.copy(scheduler)

    y = []
    for epoch in range(args.epochs):
        for i in range(num_steps):
            step = i + num_steps * epoch
            scheduler.step(step, optimizer)
            y.append(optimizer.param_groups[0]['lr'])
    print(y[0])
    print(y[-1])
    pyplot.plot(y, '.-', label='LR')
    pyplot.xlabel('step')
    pyplot.ylabel('LR')
    pyplot.grid()
    pyplot.xlim(0, args.epochs * num_steps)
    pyplot.ylim(0)
    pyplot.savefig('./weights/lr.png', dpi=200)
    pyplot.close()


class CosineLR:
    def __init__(self, args, params, num_steps):
        max_lr = params['max_lr']
        min_lr = params['min_lr']

        warmup_steps = int(max(params['warmup_epochs'] * num_steps, 100))
        decay_steps = int(args.epochs * num_steps - warmup_steps)

        warmup_lr = numpy.linspace(min_lr, max_lr, int(warmup_steps))

        decay_lr = []
        for step in range(1, decay_steps + 1):
            alpha = math.cos(math.pi * step / decay_steps)
            decay_lr.append(min_lr + 0.5 * (max_lr - min_lr) * (1 + alpha))

        self.total_lr = numpy.concatenate((warmup_lr, decay_lr))

    def step(self, step, optimizer):
        for param_group in optimizer.param_groups:
            param_group['lr'] = self.total_lr[step]


class LinearLR:
    def __init__(self, args, params, num_steps):
        max_lr = params['max_lr']
        min_lr = params['min_lr']

        warmup_steps = int(max(params['warmup_epochs'] * num_steps, 100))
        decay_steps = int(args.epochs * num_steps - warmup_steps)

        warmup_lr = numpy.linspace(min_lr, max_lr, int(warmup_steps), endpoint=False)
        decay_lr = numpy.linspace(max_lr, min_lr, decay_steps)

        self.total_lr = numpy.concatenate((warmup_lr, decay_lr))

    def step(self, step, optimizer):
        for param_group in optimizer.param_groups:
            param_group['lr'] = self.total_lr[step]


class EMA:
    """
    Updated Exponential Moving Average (EMA) from https://github.com/rwightman/pytorch-image-models
    Keeps a moving average of everything in the model state_dict (parameters and buffers)
    For EMA details see https://www.tensorflow.org/api_docs/python/tf/train/ExponentialMovingAverage
    """

    def __init__(self, model, decay=0.9999, tau=2000, updates=0):
        # Create EMA
        self.ema = copy.deepcopy(model).eval()  # FP32 EMA
        self.updates = updates  # number of EMA updates
        # decay exponential ramp (to help early epochs)
        self.decay = lambda x: decay * (1 - math.exp(-x / tau))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        if hasattr(model, 'module'):
            model = model.module
        # Update EMA parameters
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)

            msd = model.state_dict()  # model state_dict
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1 - d) * msd[k].detach()


class AverageMeter:
    def __init__(self):
        self.num = 0
        self.sum = 0
        self.avg = 0

    def update(self, v, n):
        if not math.isnan(float(v)):
            self.num = self.num + n
            self.sum = self.sum + v * n
            self.avg = self.sum / self.num


class Assigner(torch.nn.Module):
    def __init__(self, nc=80, top_k=13, alpha=1.0, beta=6.0, eps=1E-9):
        super().__init__()
        self.top_k = top_k
        self.nc = nc
        self.alpha = alpha
        self.beta = beta
        self.eps = eps

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        batch_size = pd_scores.size(0)
        num_max_boxes = gt_bboxes.size(1)

        if num_max_boxes == 0:
            device = gt_bboxes.device
            return (torch.zeros_like(pd_bboxes).to(device),
                    torch.zeros_like(pd_scores).to(device),
                    torch.zeros_like(pd_scores[..., 0]).to(device))

        num_anchors = anc_points.shape[0]
        shape = gt_bboxes.shape
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)
        mask_in_gts = torch.cat((anc_points[None] - lt, rb - anc_points[None]), dim=2)
        mask_in_gts = mask_in_gts.view(shape[0], shape[1], num_anchors, -1).amin(3).gt_(self.eps)
        na = pd_bboxes.shape[-2]
        gt_mask = (mask_in_gts * mask_gt).bool()  # b, max_num_obj, h*w
        overlaps = torch.zeros([batch_size, num_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        bbox_scores = torch.zeros([batch_size, num_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device)

        ind = torch.zeros([2, batch_size, num_max_boxes], dtype=torch.long)  # 2, b, max_num_obj
        ind[0] = torch.arange(end=batch_size).view(-1, 1).expand(-1, num_max_boxes)  # b, max_num_obj
        ind[1] = gt_labels.squeeze(-1)  # b, max_num_obj
        bbox_scores[gt_mask] = pd_scores[ind[0], :, ind[1]][gt_mask]  # b, max_num_obj, h*w

        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, num_max_boxes, -1, -1)[gt_mask]
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[gt_mask]
        overlaps[gt_mask] = compute_iou(gt_boxes, pd_boxes).squeeze(-1).clamp_(0)

        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta)

        top_k_mask = mask_gt.expand(-1, -1, self.top_k).bool()
        top_k_metrics, top_k_indices = torch.topk(align_metric, self.top_k, dim=-1, largest=True)
        if top_k_mask is None:
            top_k_mask = (top_k_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(top_k_indices)
        top_k_indices.masked_fill_(~top_k_mask, 0)

        mask_top_k = torch.zeros(align_metric.shape, dtype=torch.int8, device=top_k_indices.device)
        ones = torch.ones_like(top_k_indices[:, :, :1], dtype=torch.int8, device=top_k_indices.device)
        for k in range(self.top_k):
            mask_top_k.scatter_add_(-1, top_k_indices[:, :, k:k + 1], ones)
        mask_top_k.masked_fill_(mask_top_k > 1, 0)
        mask_top_k = mask_top_k.to(align_metric.dtype)
        mask_pos = mask_top_k * mask_in_gts * mask_gt

        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, num_max_boxes, -1)
            max_overlaps_idx = overlaps.argmax(1)

            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)

            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()
            fg_mask = mask_pos.sum(-2)
        target_gt_idx = mask_pos.argmax(-2)

        # Assigned target
        index = torch.arange(end=batch_size, dtype=torch.int64, device=gt_labels.device)[..., None]
        target_index = target_gt_idx + index * num_max_boxes
        target_labels = gt_labels.long().flatten()[target_index]

        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_index]

        # Assigned target scores
        target_labels.clamp_(0)

        target_scores = torch.zeros((target_labels.shape[0], target_labels.shape[1], self.nc),
                                    dtype=torch.int64,
                                    device=target_labels.device)
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)

        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.nc)
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0)

        # Normalize
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)).amax(-2).unsqueeze(-1)
        target_scores = target_scores * norm_align_metric

        return target_bboxes, target_scores, fg_mask.bool()


class QFL(torch.nn.Module):
    def __init__(self, beta=2.0):
        super().__init__()
        self.beta = beta
        self.bce_loss = torch.nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, outputs, targets):
        bce_loss = self.bce_loss(outputs, targets)
        return torch.pow(torch.abs(targets - outputs.sigmoid()), self.beta) * bce_loss


class VFL(torch.nn.Module):
    def __init__(self, alpha=0.75, gamma=2.00, iou_weighted=True):
        super().__init__()
        assert alpha >= 0.0
        self.alpha = alpha
        self.gamma = gamma
        self.iou_weighted = iou_weighted
        self.bce_loss = torch.nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, outputs, targets):
        assert outputs.size() == targets.size()
        targets = targets.type_as(outputs)

        if self.iou_weighted:
            focal_weight = targets * (targets > 0.0).float() + \
                           self.alpha * (outputs.sigmoid() - targets).abs().pow(self.gamma) * \
                           (targets <= 0.0).float()

        else:
            focal_weight = (targets > 0.0).float() + \
                           self.alpha * (outputs.sigmoid() - targets).abs().pow(self.gamma) * \
                           (targets <= 0.0).float()

        return self.bce_loss(outputs, targets) * focal_weight


class BoxLoss(torch.nn.Module):
    def __init__(self, dfl_ch):
        super().__init__()
        self.dfl_ch = dfl_ch

    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask):
        # IoU loss
        weight = torch.masked_select(target_scores.sum(-1), fg_mask).unsqueeze(-1)
        iou = compute_iou(pred_bboxes[fg_mask], target_bboxes[fg_mask])
        loss_box = ((1.0 - iou) * weight).sum() / target_scores_sum

        # DFL loss
        a, b = target_bboxes.chunk(2, -1)
        target = torch.cat((anchor_points - a, b - anchor_points), -1)
        target = target.clamp(0, self.dfl_ch - 0.01)
        loss_dfl = self.df_loss(pred_dist[fg_mask].view(-1, self.dfl_ch + 1), target[fg_mask])
        loss_dfl = (loss_dfl * weight).sum() / target_scores_sum

        return loss_box, loss_dfl

    @staticmethod
    def df_loss(pred_dist, target):
        # Distribution Focal Loss (DFL)
        # https://ieeexplore.ieee.org/document/9792391
        tl = target.long()  # target left
        tr = tl + 1  # target right
        wl = tr - target  # weight left
        wr = 1 - wl  # weight right
        left_loss = cross_entropy(pred_dist, tl.view(-1), reduction='none').view(tl.shape)
        right_loss = cross_entropy(pred_dist, tr.view(-1), reduction='none').view(tl.shape)
        return (left_loss * wl + right_loss * wr).mean(-1, keepdim=True)


class ComputeLoss:
    def __init__(self, model, params):
        if hasattr(model, 'module'):
            model = model.module

        device = next(model.parameters()).device

        m = model.head  # Head() module

        self.params = params
        self.stride = m.stride
        self.nc = m.nc
        self.no = m.no
        self.reg_max = m.ch
        self.device = device

        self.box_loss = BoxLoss(m.ch - 1).to(device)
        self.cls_loss = torch.nn.BCEWithLogitsLoss(reduction='none')
        self.assigner = Assigner(nc=self.nc, top_k=10, alpha=0.5, beta=6.0)

        self.project = torch.arange(m.ch, dtype=torch.float, device=device)

    def box_decode(self, anchor_points, pred_dist):
        b, a, c = pred_dist.shape
        pred_dist = pred_dist.view(b, a, 4, c // 4)
        pred_dist = pred_dist.softmax(3)
        pred_dist = pred_dist.matmul(self.project.type(pred_dist.dtype))
        lt, rb = pred_dist.chunk(2, -1)
        x1y1 = anchor_points - lt
        x2y2 = anchor_points + rb
        return torch.cat(tensors=(x1y1, x2y2), dim=-1)

    def __call__(self, outputs, targets):
        x = torch.cat([i.view(outputs[0].shape[0], self.no, -1) for i in outputs], dim=2)
        pred_distri, pred_scores = x.split(split_size=(self.reg_max * 4, self.nc), dim=1)

        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()

        data_type = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        input_size = torch.tensor(outputs[0].shape[2:], device=self.device, dtype=data_type) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(outputs, self.stride, offset=0.5)

        idx = targets['idx'].view(-1, 1)
        cls = targets['cls'].view(-1, 1)
        box = targets['box']

        targets = torch.cat((idx, cls, box), dim=1).to(self.device)
        if targets.shape[0] == 0:
            gt = torch.zeros(batch_size, 0, 5, device=self.device)
        else:
            i = targets[:, 0]
            _, counts = i.unique(return_counts=True)
            counts = counts.to(dtype=torch.int32)
            gt = torch.zeros(batch_size, counts.max(), 5, device=self.device)
            for j in range(batch_size):
                matches = i == j
                n = matches.sum()
                if n:
                    gt[j, :n] = targets[matches, 1:]
            x = gt[..., 1:5].mul_(input_size[[1, 0, 1, 0]])
            y = torch.empty_like(x)
            dw = x[..., 2] / 2  # half-width
            dh = x[..., 3] / 2  # half-height
            y[..., 0] = x[..., 0] - dw  # top left x
            y[..., 1] = x[..., 1] - dh  # top left y
            y[..., 2] = x[..., 0] + dw  # bottom right x
            y[..., 3] = x[..., 1] + dh  # bottom right y
            gt[..., 1:5] = y
        gt_labels, gt_bboxes = gt.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)

        pred_bboxes = self.box_decode(anchor_points, pred_distri)
        assigned_targets = self.assigner(pred_scores.detach().sigmoid(),
                                         (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
                                         anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_bboxes, target_scores, fg_mask = assigned_targets
        # num_pos = fg_mask.sum().item()
        # print(f"DEBUG -> Targets matched: {num_pos}")

        target_scores_sum = max(target_scores.sum(), 1)

        loss_cls = self.cls_loss(pred_scores, target_scores.to(data_type)).sum() / target_scores_sum  # BCE

        # Box loss
        loss_box = torch.zeros(1, device=self.device)
        loss_dfl = torch.zeros(1, device=self.device)
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            loss_box, loss_dfl = self.box_loss(pred_distri,
                                               pred_bboxes,
                                               anchor_points,
                                               target_bboxes,
                                               target_scores,
                                               target_scores_sum, fg_mask)

        loss_box *= self.params['box']  # box gain
        loss_cls *= self.params['cls']  # cls gain
        loss_dfl *= self.params['dfl']  # dfl gain

        return loss_box, loss_cls, loss_dfl

In [ ]:
def decode_boxes(raw_preds, strides=[8, 16, 32], nc=80, img_size=640):
    """
    Decodes flattened YOLOv8 outputs. 
    Expects raw_preds as a single tensor [batch, 144, 8400] 
    or a list of flattened tensors.
    """
    device = raw_preds[0].device if isinstance(raw_preds, list) else raw_preds.device
    
    # If raw_preds is a list of 3 scales, flatten and concat them first
    if isinstance(raw_preds, list):
        # Flatten H*W for each scale and concat on the last dim
        x = torch.cat([scales.view(scales.shape[0], scales.shape[1], -1) for scales in raw_preds], dim=2)
    else:
        x = raw_preds

    # x shape is now [batch, 144, 8400]
    batch_size, channels, num_anchors = x.shape
    
    # 1. Split Box (64) and Class (80)
    raw_dist = x[:, :64, :]    # [batch, 64, 8400]
    cls_scores = x[:, 64:, :].sigmoid() # [batch, 80, 8400]

    # 2. Generate Anchors and Strides for the whole map
    # We need a grid that matches the 8400 anchors (80x80 + 40x40 + 20x20)
    anchor_points = []
    stride_tensor = []
    for s in strides:
        h = w = img_size // s
        grid_y, grid_x = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
        # [H*W, 2] -> (x, y)
        grid = torch.stack((grid_x, grid_y), dim=-1).to(device).float().view(-1, 2)
        anchor_points.append(grid + 0.5) # Center of the pixel
        stride_tensor.append(torch.full((h * w, 1), s, device=device))
    
    anchors = torch.cat(anchor_points, dim=0) # [8400, 2]
    strides_flat = torch.cat(stride_tensor, dim=0) # [8400, 1]

    # 3. DFL Decoding (Distances)
    # Convert 16 bins per side to 1 distance value
    raw_dist = raw_dist.view(batch_size, 4, 16, num_anchors).softmax(2)
    range_tensor = torch.arange(16).view(1, 1, 16, 1).to(device).float()
    dist = (raw_dist * range_tensor).sum(2) # [batch, 4, 8400]

    # 4. Decode to XYXY (Normalized 0-1)
    # dist is [left, top, right, bottom]
    x1 = (anchors[:, 0] - dist[:, 0]) * strides_flat.T
    y1 = (anchors[:, 1] - dist[:, 1]) * strides_flat.T
    x2 = (anchors[:, 0] + dist[:, 2]) * strides_flat.T
    y2 = (anchors[:, 1] + dist[:, 3]) * strides_flat.T
    
    decoded_boxes = torch.stack([x1, y1, x2, y2], dim=1) # [batch, 4, 8400]
    
    # 5. Combine and Transpose for NMS: [batch, 8400, 84]
    return torch.cat([decoded_boxes, cls_scores], dim=1).transpose(1, 2)

def validate(model, loader, device, conf_thres=0.001, iou_thres=0.6):
    print(f"Confidence thres: {conf_thres} | IoU thres: {iou_thres}")
    model.eval()
    stats = []
    iou_v = torch.linspace(0.5, 0.95, 10).to(device)
    
    # --- 1. DEFINE CLASS MAPPING ---
    train_names = params['names']
    val_names = {0: 'bottle', 1: 'person', 2: 'hot dog', 3: 'toilet', 4: 'bowl', 5: 'sink', 6: 'vase', 7: 'potted plant', 8: 'cup', 9: 'apple', 10: 'microwave', 11: 'oven', 12: 'airplane', 13: 'clock', 14: 'dining table', 15: 'pizza', 16: 'baseball bat', 17: 'baseball glove', 18: 'traffic light', 19: 'skis', 20: 'car', 21: 'fire hydrant', 22: 'kite', 23: 'sports ball', 24: 'chair', 25: 'tennis racket', 26: 'bus', 27: 'frisbee', 28: 'wine glass', 29: 'elephant', 30: 'train', 31: 'toothbrush', 32: 'cat', 33: 'book', 34: 'couch', 35: 'skateboard', 36: 'bird', 37: 'truck', 38: 'dog', 39: 'fork', 40: 'spoon', 41: 'knife', 42: 'tie', 43: 'motorcycle', 44: 'tv', 45: 'bicycle', 46: 'suitcase', 47: 'handbag', 48: 'laptop', 49: 'remote', 50: 'surfboard', 51: 'banana', 52: 'backpack', 53: 'bed', 54: 'boat', 55: 'cow', 56: 'snowboard', 57: 'giraffe', 58: 'bench', 59: 'umbrella', 60: 'cake', 61: 'teddy bear', 62: 'horse', 63: 'cell phone', 64: 'zebra', 65: 'donut', 66: 'orange', 67: 'keyboard', 68: 'broccoli', 69: 'sandwich', 70: 'carrot', 71: 'sheep', 72: 'scissors', 73: 'parking meter', 74: 'mouse', 75: 'stop sign', 76: 'bear', 77: 'refrigerator', 78: 'hair drier', 79: 'toaster'}

    # Build val_id -> train_id map
    inv_train = {v: k for k, v in train_names.items()}
    id_map = torch.tensor([inv_train[val_names[i]] for i in range(80)]).to(device)

    pbar = tqdm(loader, desc="Validating", leave=False)
    
    with torch.no_grad():
        for imgs, targets in pbar:
            imgs = imgs.to(device).float() / 255.0
            
            # Map Validation IDs to Training IDs
            targets['cls'] = id_map[targets['cls'].to(device).long()]

            # Inference
            raw_outputs = model(imgs) 
            decoded_outputs = decode_boxes(raw_outputs, strides=[8, 16, 32], nc=80)
            preds = non_max_suppression(decoded_outputs, conf_thres, iou_thres)

            # import matplotlib.pyplot as plt
            # import matplotlib.patches as patches
            
            # # Only do this for the very first image
            # img = imgs[0].cpu().permute(1, 2, 0).numpy()
            # plt.imshow(img)
            # ax = plt.gca()
            
            # # Draw Ground Truth in Green
            # for box in t_box_xyxy.cpu():
            #     rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], linewidth=2, edgecolor='g', facecolor='none')
            #     ax.add_patch(rect)
            
            # # Draw Predictions in Red
            # for p in preds[0].cpu():
            #     if p[4] > 0.01: # threshold to see something
            #         rect = patches.Rectangle((p[0], p[1]), p[2]-p[0], p[3]-p[1], linewidth=1, edgecolor='r', facecolor='none')
            #         ax.add_patch(rect)
            
            # plt.show()
            # break # Stop after one image

            if len(preds[0]) > 0:
                print(f"DEBUG: First Pred Box: {preds[0][0, :4]}") 
                # If x2 < x1 or y2 < y1, your format is WRONG. 
                # Example: [320, 240, 10, 15] is CXCYWH. [315, 232, 325, 247] is XYXY.

            for i, pred in enumerate(preds):
                img_mask = (targets['idx'] == i)
                t_cls = targets['cls'][img_mask].to(device)
                t_box = targets['box'][img_mask].to(device)
                
                h, w = imgs.shape[2:]
                t_box_xyxy = wh2xy(t_box, w, h)
                
                if pred.shape[0] == 0:
                    if t_cls.shape[0] > 0:
                        stats.append((torch.zeros(0, 10, dtype=torch.bool).cpu(), 
                                      torch.tensor([]).cpu(), torch.tensor([]).cpu(), t_cls.cpu()))
                    continue

                # Compute Metrics
                t_formatted = torch.cat((t_cls.view(-1, 1), t_box_xyxy), 1)
                correct = compute_metric(pred, t_formatted, iou_v)

                # Append to stats (all to CPU)
                stats.append((
                    correct.cpu(), 
                    pred[:, 4].cpu(), 
                    pred[:, 5].cpu(), 
                    t_cls.cpu()
                ))

    # Compute Final Results
    if len(stats) > 0:
        stats_cpu = [torch.cat(x, 0).numpy() for x in zip(*stats)]
        if stats_cpu[0].any():
            tp, conf, p_cls, t_cls_all = stats_cpu
            # Unpack the 6 values from your specific compute_ap
            _, _, p, r, map50, mean_ap = compute_ap(tp, conf, p_cls, t_cls_all)
            return p, r, map50, mean_ap
            
    return 0, 0, 0, 0

In [ ]:
import os
import torch
from torch.cuda.amp import autocast, GradScaler

from time import time, strftime, gmtime
from tqdm import tqdm
import sys

# Configuration
weights_dir = './weights'
last_path = os.path.join(weights_dir, 'last.pt')
best_path = os.path.join(weights_dir, 'best.pt')
os.makedirs(weights_dir, exist_ok=True)

# Device and Scaler Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scaler = GradScaler()

# 1. Initialize fresh model and optimizer
model = CustomYolo(version='s').to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.05)

current_lr = optimizer.param_groups[0]['lr']
print(f"Current LR: {current_lr:.6f}")

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=0.00001)
# Compute actual strides (don't leave them as 0)
with torch.no_grad():
    dummy = torch.zeros(1, 3, 640, 640).to(device)
    outputs = model(dummy)
    model.head.stride = torch.tensor([640 / x.shape[-2] for x in outputs]).to(device)
    print(f"Verified Strides: {model.head.stride}") # Should be [8, 16, 32]
criterion = ComputeLoss(model, params)
start_epoch = 0
best_loss = float('inf')

# 2. Check for existing checkpoint to resume
if os.path.exists(best_path):
    print(f"Found checkpoint: {best_path}. Resuming training...")
    try:
        # Using weights_only=False because we are loading a custom dict with params
        checkpoint = torch.load(best_path, map_location=device, weights_only=False)
        
        # Restore model and optimizer states
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        
        # Restore training metadata
        start_epoch = checkpoint['epoch'] + 1
        # print(checkpoint['best_lossl'])
        best_loss = checkpoint.get('best_loss', float('inf'))
        
        print(f"Resuming from Epoch {start_epoch} with Best Loss: {best_loss:.4f}")
    except Exception as e:
        print(f"Failed to resume: {e}. Starting from scratch.")
else:
    print("No checkpoint found. Starting fresh training.")

num_epochs = 150
num_batches = 100

log_file = 'losshistory.txt'
if not os.path.exists(log_file):
    with open(log_file, 'w') as f:
        # CSV style header for easy parsing later
        f.write("epoch,batch,total_loss,box_loss,cls_loss,dfl_loss\n")
os.makedirs('./weights', exist_ok=True)
# best_loss =4.1597
best_map = 0.0

In [ ]:
# Training Loop
start_training_time = time()
for epoch in range(start_epoch, num_epochs):
    epoch_start_time = time()
    model.train()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current LR: {current_lr:.6f}")
    for param_group in optimizer.param_groups:
        param_group['lr'] = 0.0001

    print(f"🚀 Learning rate manually updated to: {optimizer.param_groups[0]['lr']}")

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}", file=sys.stdout)
    epoch_losses = {'total': 0, 'box': 0, 'cls': 0, 'dfl': 0}
    
    for i, (imgs, targets) in pbar:
        # Move data to GPU and normalize pixels to [0, 1]
        imgs = imgs.to(device).float() / 255.0
        targets = {k: v.to(device) for k, v in targets.items()}
        
        optimizer.zero_grad()

        # Autocast forward pass (FP16/BF16)
        with autocast():
            outputs = model(imgs)
            # if i == 0: # Check the very first batch
            #     print("\n--- COORDINATE SANITY CHECK ---")
            #     print(f"Target Box Sample (first 5): \n{targets['box'][:5]}")
            #     print(f"Target Box Max Value: {targets['box'].max().item()}")
                
            #     if targets['box'].max().item() > 1.0:
            #         print("⚠️ WARNING: Target boxes are > 1. This means they are in PIXELS.")
            #         print("YOLO Loss usually expects NORMALIZED (0-1) coordinates.")
            #     else:
            #         print("✅ Target boxes look normalized (0-1).")
                
            #     # Check model output scale
            #     raw_preds = outputs[0] if isinstance(outputs, list) else outputs
            #     print(f"Model Output Shape: {raw_preds.shape}")
            #     print(f"Model Output Max/Min: {raw_preds.max().item():.2f} / {raw_preds.min().item():.2f}")
            loss_box, loss_cls, loss_dfl = criterion(outputs, targets)
            loss = loss_box + loss_cls + loss_dfl

        # Scaled Backward Pass to prevent 'gradient underflow' which happens in mixed precision
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update running losses for the progress bar
        epoch_losses['total'] += loss.item()
        epoch_losses['box'] += loss_box.item()
        epoch_losses['cls'] += loss_cls.item()
        epoch_losses['dfl'] += loss_dfl.item()

        if i % 10 == 0:
            status = (f"Batch {i}/{len(train_loader)} | "
                      f"Total Loss: {loss.item():.4f} | "
                      f"Box Loss: {loss_box.item():.4f} | "
                      f"Class Loss: {loss_cls.item():.4f} | "
                      f"DFL Loss: {loss_dfl.item():.4f}")
            pbar.write(status)
            with open(log_file, 'a') as f:
                log_string = (f"{epoch},{i},{loss.item():.6f},"
                      f"{loss_box.item():.6f},{loss_cls.item():.6f},{loss_dfl.item():.6f}\n")
                f.write(log_string)
            # print(f"Epoch {epoch} | Batch {i} | Total Loss: {loss.item():.4f} "
            #       f"(Box: {loss_box.item():.2f}, Cls: {loss_cls.item():.2f}, DFL: {loss_dfl.item():.2f})")
        
        pbar.set_postfix({'loss': f"{loss.item():.2f}"})
    epoch_duration = time() - epoch_start_time
    avg_epoch_loss = epoch_losses['total'] / len(train_loader)
    epoch_box_loss = epoch_losses['box'] / len(train_loader)
    epoch_cls_loss = epoch_losses['cls'] / len(train_loader)
    epoch_dfl_loss = epoch_losses['dfl'] / len(train_loader)

    # ETA Calculation
    epochs_remaining = num_epochs - (epoch + 1)
    eta_seconds = epochs_remaining * epoch_duration
    eta_str = strftime("%H:%M:%S", gmtime(eta_seconds))

    print(f"\nSummary Epoch {epoch}: Loss {avg_epoch_loss:.4f} | Time: {epoch_duration}s | ETA: {eta_str}")
    print(f"\nAvg. box loss: {epoch_box_loss} | Avg. cls loss: {epoch_cls_loss} | Avg. dfl loss: {epoch_dfl_loss}")
    
    # scheduler.step()
    
    # Checkpoint Dictionary
    ckpt = {
        'epoch': epoch,
        'best_loss': best_loss,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'params': params
    }

    # Save Last Checkpoint
    torch.save(ckpt, last_path)
    # Save Best Checkpoint (Only if loss improved)
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        ckpt['best_loss'] = best_loss # Update best loss in dict
        torch.save(ckpt, os.path.join(weights_dir, 'best.pt'))
        print(f"New Best Model saved at Epoch {epoch} with Loss: {best_loss:.4f}")

    # if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
    #     p, r, map50, mAP = validate(model, val_loader, device)
        
    #     # Log metrics to console
    #     print(f"\nValidation Results (Epoch {epoch}):")
    #     print(f"Precision: {p:.4f} | Recall: {r:.4f} | mAP50: {map50:.4f} | mAP50-95: {mAP:.4f}\n")

    #     # Log to your permanent file
    #     with open('losshistory.txt', 'a') as f:
    #         f.write(f"{epoch},VALIDATION,{p:.4f},{r:.4f},{map50:.4f},{mAP:.4f}\n")

    #     # 5. Checkpoint based on mAP (Standard for Detection)
    #     if mAP > best_map:
    #         best_map = mAP
    #         ckpt = {
    #             'epoch': epoch,
    #             'model': model.state_dict(),
    #             'optimizer': optimizer.state_dict(),
    #             'mAP': best_map
    #         }
    #         torch.save(ckpt, './weights/best.pt')
    #         print(f"New Best mAP: {best_map:.4f} - Saved Weights")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch} | Current LR: {current_lr:.6f}")


In [ ]:
p, r, map50, mAP = validate(model, subset_loader, device, conf_thres=0.01, iou_thres=0.35)

# Log metrics to console
# print(f"\nValidation Results (Epoch {epoch}):")
print(f"Precision: {p:.4f} | Recall: {r:.4f} | mAP50: {map50:.4f} | mAP50-95: {mAP:.4f}\n")

In [ ]:
current_lr = optimizer.param_groups[0]['lr']
print(f"Current LR: {current_lr:.8f}")
print(best_loss)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

model.eval()
with torch.no_grad():
    # 1. Grab exactly one batch
    imgs, targets = next(iter(subset_loader))
    imgs = imgs.to(device).float() / 255.0
    
    # 2. Run your model
    raw_outputs = model(imgs)
    decoded_outputs = decode_boxes(raw_outputs, confidence_threshold=[8, 16, 32], nc=80)
    # Use a very low threshold to see EVERYTHING the model is thinking
    preds = non_max_suppression(decoded_outputs, confidence_threshold=0.01, iou_threshold=0.3)

    # 3. Plot the first image of the batch
    img = imgs[0].cpu().permute(1, 2, 0).numpy()
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    ax = plt.gca()

    # 4. Draw Ground Truth (Green)
    img_mask = (targets['idx'] == 0)
    t_box = targets['box'][img_mask] 
    t_box_pixel = wh2xy(t_box, 640, 640).cpu() # Move to CPU here

    print(f"Drawing {len(t_box_pixel)} Ground Truth boxes...")
    for box in t_box_pixel:
        x1, y1, x2, y2 = box[0].item(), box[1].item(), box[2].item(), box[3].item()
        w, h = x2 - x1, y2 - y1
        rect = patches.Rectangle((x1, y1), w, h, linewidth=3, edgecolor='g', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, 'GT', color='green', fontsize=12, fontweight='bold', backgroundcolor='black')

    # 5. Draw Predictions (Red)
    p_batch = preds[0].cpu() # Move entire prediction batch to CPU
    print(f"Found {len(p_batch)} total predictions after NMS.")
    
    for p in p_batch:
        conf = p[4].item()
        if conf > 0.1: # Only show reasonably confident ones to avoid clutter
            x1, y1, x2, y2 = p[0].item(), p[1].item(), p[2].item(), p[3].item()
            w, h = x2 - x1, y2 - y1
            rect = patches.Rectangle((x1, y1), w, h, linewidth=1, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1, f'{conf:.2f}', color='red', fontsize=8, backgroundcolor='white')

    plt.title("Visual Check: Green=GT, Red=Pred")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch.nn.functional as F

def plot_fresh_start(model, loader, device, reg_max=16):
    model.eval()
    # 1. Get one batch
    imgs, targets = next(iter(loader))
    imgs = imgs.to(device).float() / 255.0
    
    with torch.no_grad():
        # raw_output is a list of 3 scales: [32, 144, 80, 80], [32, 144, 40, 40], [32, 144, 20, 20]
        raw_output = model(imgs)

    # 3. Setup Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(imgs[0].cpu().permute(1, 2, 0).numpy())
    
    # --- DECODING LOGIC ---
    strides = [8, 16, 32]
    all_preds = []

    for i, stride in enumerate(strides):
        x = raw_output[i][0]  # Take first image in batch: [144, H, W]
        h, w = x.shape[1:]
        
        # Split Box (64) and Class (80)
        box_dist = x[:64, :, :] # [64, H, W]
        cls_logits = x[64:, :, :] # [80, H, W]
        
        # Decode DFL (The 16.19 values you saw)
        # Reshape to [4, 16, H, W], softmax over the 16 bins
        box_dist = box_dist.view(4, reg_max, h, w).softmax(1)
        # Weighted sum of bins to get distance
        proj = torch.arange(reg_max).float().to(device).view(1, reg_max, 1, 1)
        dist = (box_dist * proj).sum(1) # [4, H, W] -> [left, top, right, bottom]
        
        # Create Grid
        y, x_grid = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
        x_grid, y = x_grid.to(device), y.to(device)
        
        # Calculate XYXY in Pixels
        # (grid_x - left) * stride ... (grid_x + right) * stride
        x1 = (x_grid + 0.5 - dist[0]) * stride
        y1 = (y + 0.5 - dist[1]) * stride
        x2 = (x_grid + 0.5 + dist[2]) * stride
        y2 = (y + 0.5 + dist[3]) * stride
        
        # Get Max Class Score
        conf, clss = cls_logits.sigmoid().max(0) # [H, W]
        
        # Stack into [H*W, 6] -> x1, y1, x2, y2, conf, class
        scale_preds = torch.stack([x1, y1, x2, y2, conf, clss.float()], dim=0)
        all_preds.append(scale_preds.view(6, -1).T)

    # Combine all scales
    final_preds = torch.cat(all_preds, dim=0)
    
    # --- DRAWING ---
    # 4. Draw Ground Truth (Green)
    img_idx = 0
    gt_mask = targets['idx'] == img_idx
    for gt_box in targets['box'][gt_mask]:
        # Convert CXCYWH Normalized -> XYXY Pixels
        cx, cy, bw, bh = gt_box
        gx1, gy1 = (cx - bw/2) * 640, (cy - bh/2) * 640
        gx2, gy2 = (cx + bw/2) * 640, (cy + bh/2) * 640
        ax.add_patch(patches.Rectangle((gx1.item(), gy1.item()), (gx2-gx1).item(), (gy2-gy1).item(), 
                                       linewidth=3, edgecolor='lime', facecolor='none'))

    # 5. Draw Predictions (Red) - Filter by 0.3 conf
    for p in final_preds:
        if p[4] > 0.3:
            px1, py1, px2, py2, pconf = p[:5].cpu().numpy()
            ax.add_patch(patches.Rectangle((px1, py1), px2-px1, py2-py1, 
                                           linewidth=1, edgecolor='red', facecolor='none'))
            ax.text(px1, py1, f"{pconf:.2f}", color='white', fontsize=8, backgroundcolor='red')

    plt.title("Green=GT | Red=Pred (Conf > 0.3)")
    plt.show()

plot_fresh_start(model, subset_loader, device)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

def plot_fresh_start(model, loader, device, reg_max=16):
    model.eval()
    imgs, targets = next(iter(loader))
    imgs = imgs.to(device).float() / 255.0
    
    with torch.no_grad():
        raw_output = model(imgs)

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(imgs[0].cpu().permute(1, 2, 0).numpy())
    
    all_preds = []
    
    # Handle list of scales vs. single tensor
    if isinstance(raw_output, (list, tuple)):
        feats = raw_output
    else:
        # If it's [Batch, 144, 8400], we treat it as one giant feature map
        feats = [raw_output]

    # For a list of scales like [32, 144, 80, 80], [32, 144, 40, 40], etc.
    strides = [8, 16, 32]
    
    for i, x in enumerate(feats):
        # Shape of x: [Batch, Channels, H, W] or [Batch, Channels, Anchors]
        x = x[0] # Take first image in batch
        
        if len(x.shape) == 3: # Case: [Channels, H, W]
            c, h, w = x.shape
            stride = strides[i] if i < len(strides) else 8
            
            # 1. DFL Decoding
            box_dist = x[:64, :, :].view(4, reg_max, h, w).softmax(1)
            proj = torch.arange(reg_max).float().to(device).view(1, reg_max, 1, 1)
            dist = (box_dist * proj).sum(1) # [4, H, W]
            
            # 2. Grid Generation
            y, x_grid = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
            x_grid, y = x_grid.to(device), y.to(device)
            
            # 3. Transform to Pixels
            x1 = (x_grid + 0.5 - dist[0]) * stride
            y1 = (y + 0.5 - dist[1]) * stride
            x2 = (x_grid + 0.5 + dist[2]) * stride
            y2 = (y + 0.5 + dist[3]) * stride
            
            # 4. Class Scores
            conf, clss = x[64:, :, :].sigmoid().max(0)
            
            scale_preds = torch.stack([x1, y1, x2, y2, conf, clss.float()], dim=0)
            all_preds.append(scale_preds.view(6, -1).T)
            
        else: # Case: [Channels, Anchors] (already flattened)
            print("Detected flattened output. Using manual anchor logic...")
            # If your model already returns flattened [144, 8400]
            # We'd need the anchor/stride tensor we built in decode_boxes
            # Let's skip drawing for now and just print info to debug
            print(f"Flattened Shape: {x.shape}")

    if not all_preds:
        print("No predictions generated. Check model output format.")
        return

    final_preds = torch.cat(all_preds, dim=0)
    
    # --- DRAWING ---
    img_idx = 0
    gt_mask = targets['idx'] == img_idx
    for gt_box in targets['box'][gt_mask]:
        cx, cy, bw, bh = gt_box
        gx1, gy1 = (cx - bw/2) * 640, (cy - bh/2) * 640
        gx2, gy2 = (cx + bw/2) * 640, (cy + bh/2) * 640
        ax.add_patch(patches.Rectangle((gx1.item(), gy1.item()), (gx2-gx1).item(), (gy2-gy1).item(), 
                                       linewidth=3, edgecolor='lime', facecolor='none'))

    # Draw Red Boxes (Predictions)
    draw_count = 0
    for p in final_preds:
        if p[4] > 0.2: # Confidence threshold
            px1, py1, px2, py2, pconf = p[:5].cpu().numpy()
            ax.add_patch(patches.Rectangle((px1, py1), px2-px1, py2-py1, 
                                           linewidth=1, edgecolor='red', facecolor='none'))
            draw_count += 1
            if draw_count > 50: break # Don't crash the UI with 8000 boxes

    plt.title(f"Visual Check | Green=GT | Red=Pred (Found {draw_count} visible)")
    plt.show()

plot_fresh_start(model, subset_loader, device)

In [ ]:
import os
print(os.getcwd())
import torch
import gc

def clear_vram():
    # Delete the model and criterion if they exist to free references
    global model, criterion, optimizer, train_loader
    
    # Clear out the cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Force garbage collection
    gc.collect()
    
    # Final cache clear
    torch.cuda.empty_cache()
    
    print("VRAM cleared.")

clear_vram()

In [ ]:
import torch
import cv2
import numpy as np

def run_detection(img_path, model_version='s', weights_path=None, conf_threshold=0.25):
    # 1. Setup Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 2. Initialize Model (ensure all classes from the notebook are defined)
    # The 'CustomYolo' class from the notebook wraps Backbone, Neck, and Head.
    model = CustomYolo(version=model_version).to(device)
    
    # 3. Load Trained Weights
    if weights_path:
        model.load_state_dict(torch.load(weights_path, map_local=device))
    model.eval()

    # 4. Preprocess Image
    img0 = cv2.imread(img_path)
    # YOLO typically expects 640x640 input based on the notebook's sanity checks
    img = cv2.resize(img0, (640, 640)) 
    img = img.transpose((2, 0, 1))  # HWC to CHW
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(device).float()
    img /= 255.0  # Normalize to [0, 1]
    if img.ndimension() == 3:
        img = img.unsqueeze(0) # Add batch dimension

    # 5. Inference
    with torch.no_grad():
        # The model's forward pass returns [box_coords, class_probs]
        # output shape: [batch, 4 + num_classes, num_anchors]
        predictions = model(img)

    # 6. Post-processing (Simplified)
    # Note: For real use, you would typically apply Non-Maximum Suppression (NMS)
    # here to filter overlapping boxes.
    predictions = predictions[0].transpose(0, 1) # [num_anchors, 4 + num_classes]
    
    # Filter by confidence
    scores = predictions[:, 4:]
    max_scores, class_ids = torch.max(scores, dim=1)
    mask = max_scores > conf_threshold
    
    detections = predictions[mask]
    final_class_ids = class_ids[mask]
    final_scores = max_scores[mask]

    print(f"Detected {len(detections)} objects.")
    return detections, final_class_ids, final_scores

# Usage Example:
# detections, classes, scores = run_detection('your_image.jpg', weights_path='model_weights.pt')